# Daisyworld Model
### Gaia hypothesis demonstration

Daisyworld is a classic simulation of a planet whose temperature is regulated by the life forms inhabiting it. Two species of daisies (white and black) regulate planetary temperature through albedo feedback. White daisies reflect more light (high albedo) and cool the planet, while black daisies absorb more light (low albedo) and warm it. Their relative abundance shifts with solar luminosity, buffering the planet's temperature over a wide range.

## How it works

The model computes the planetary albedo as a weighted average of the areas covered by daisies and bare soil. This albedo then determines the planet's temperature.

| Variable | Description |
|:---|:---|
| $\alpha_{planet}$ | Planetary albedo (weighted average) |
| $T_{planet}$ | Average planetary temperature (K) |
| $T_{local}$ | Local temperature for each daisy species |
| $\beta$ | Growth rate based on local temperature |

The growth of each species depends on its local temperature being within a habitable range (approx. 5°C to 40°C).

## Imports

In [ ]:
from dissmodel.core import Environment
from dissmodel_sysdyn.models import Daisyworld
from dissmodel.visualization import Chart

## ⚠️ Important: instantiation order

The `Environment` must always be created **before** any model.

```
Environment  →  Daisyworld  →  Chart
```

In [ ]:
env = Environment()

## Setting up the model

The Daisyworld model accepts the following parameters:

| Parameter | Description | Default |
|:---|:---|:---|
| `sun_luminosity` | Fractional solar luminosity | 0.70 |
| `white_area` | Initial white daisy fractional area | 0.40 |
| `black_area` | Initial black daisy fractional area | 0.273 |
| `empty_area` | Initial bare-soil fractional area | 0.327 |
| `decay_rate` | Per-step mortality rate | 0.30 |

In [ ]:
from dissmodel.core import Model
from dissmodel.visualization import track_plot

_SIGMA      = 5.67e-8
_SOLAR_FLUX = 3668.0
_Q          = 2.06e9

def _daisy_growth_rate(temp_k: float) -> float:
    temp_c = temp_k - 273.0
    if 5.0 < temp_c < 40.0:
        return max(0.0, 1.0 - 0.003265 * (22.5 - temp_c) ** 2)
    return 0.0

def _planet_temp(luminosity: float, albedo: float) -> float:
    return (_SOLAR_FLUX * luminosity * (1.0 - albedo) / (4.0 * _SIGMA)) ** 0.25

def _local_temp(planet_temp: float, planet_albedo: float, daisy_albedo: float) -> float:
    return (_Q * (planet_albedo - daisy_albedo) + planet_temp ** 4) ** 0.25

@track_plot("white_area", "white")
@track_plot("black_area", "black")
@track_plot("empty_area", "gray")
@track_plot("ave_temp", "red")
class Daisyworld(Model):
    def setup(self, sun_luminosity: float = 0.70, white_area: float = 0.40, black_area: float = 0.273, empty_area: float = 0.327) -> None:
        self.sun_luminosity = sun_luminosity
        self.planet_area    = 1.0
        self.white_area     = white_area
        self.black_area     = black_area
        self.empty_area     = empty_area
        self.white_albedo   = 0.75
        self.black_albedo   = 0.25
        self.soil_albedo    = 0.50
        self.decay_rate     = 0.30
        self.ave_temp       = _planet_temp(self.sun_luminosity, self._planet_albedo())

    def _planet_albedo(self) -> float:
        return self.white_area * self.white_albedo + self.black_area * self.black_albedo + self.empty_area * self.soil_albedo

    def execute(self) -> None:
        pa = self._planet_albedo()
        self.ave_temp = _planet_temp(self.sun_luminosity, pa)
        self.white_area += self.white_area * (_daisy_growth_rate(_local_temp(self.ave_temp, pa, self.white_albedo)) * self.empty_area - self.decay_rate)
        self.black_area += self.black_area * (_daisy_growth_rate(_local_temp(self.ave_temp, pa, self.black_albedo)) * self.empty_area - self.decay_rate)
        self.empty_area = self.planet_area - (self.white_area + self.black_area)

In [ ]:
Daisyworld(sun_luminosity=1.0)

## Visualization

The chart will show the areas of white and black daisies, the empty soil, and the average planetary temperature.

In [ ]:
Chart(show_legend=True, show_grid=True, title="Daisyworld Regulation")

## Running the simulation

In [ ]:
env.run(100)

## Guided Experiments

- **Vary Luminosity**: Change `sun_luminosity` from 0.7 to 1.6. Note how the daisy populations shift to keep the temperature stable.
- **Extinction**: Try a very high or very low luminosity where daisies cannot survive.
- **Single Species**: Start with `black_area=0` or `white_area=0` and see if regulation is still possible.

## Conclusion

Daisyworld illustrates that a complex system of self-regulation (homeostasis) can emerge from the interactions between life and its environment, without any need for conscious intent.